# Task-03: SVM Cats vs Dogs Classifier — Walkthrough
**Prodigy Infotech Internship**

Implements an SVM to classify cat/dog images from the [Kaggle Dogs vs Cats dataset](https://www.kaggle.com/c/dogs-vs-cats/data).

This notebook is a thin walkthrough over the production `src/` package — it imports and calls the same
modules used by `main.py`, so nothing here duplicates pipeline logic.

**To use real data:** download `train.zip` from Kaggle, unzip, and place the `cat.N.jpg` / `dog.N.jpg`
files into `data/raw/train/`. No code changes required.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config_loader import load_config, ensure_output_dirs
from src.utils.logger import get_logger

config = load_config("../config/config.yaml")
ensure_output_dirs(config)
logger = get_logger("notebook", log_dir=config.get("paths", "logs_dir"))
logger.info("Config loaded; project root-anchored paths resolved.")

## 1. Data: synthetic generator (drop-in for real Kaggle data)

In [ ]:
from src.data.synthetic_generator import generate_synthetic_dataset

generate_synthetic_dataset(
    train_dir=config.get("paths", "train_dir"),
    num_images_per_class=config.get("synthetic_data", "num_images_per_class"),
    image_size=config.get("synthetic_data", "image_size"),
    noise_std=config.get("synthetic_data", "noise_std"),
    random_seed=config.get("project", "random_seed"),
)

## 2. Discover & load images (Kaggle filename convention: `cat.N.jpg` / `dog.N.jpg`)

In [ ]:
from src.data.data_loader import discover_train_records, load_dataset_arrays, train_val_test_split

records = discover_train_records(config.get("paths", "train_dir"))
images, labels = load_dataset_arrays(
    records,
    resize_dim=config.get("image", "resize_dim"),
    color_mode=config.get("image", "color_mode"),
)
images.shape, labels.shape

In [ ]:
splits = train_val_test_split(
    images, labels,
    test_size=config.get("split", "test_size"),
    val_size=config.get("split", "val_size"),
    stratify=config.get("split", "stratify"),
    random_seed=config.get("project", "random_seed"),
)
{k: v.shape[0] for k, v in splits.items() if k.startswith("x_")}

## 3. Preview a few samples
(The pipeline itself uses matplotlib's Agg backend throughout with no `plt.show()` calls;
here in the notebook the inline backend renders normally for visual inspection.)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, img, lbl in zip(axes, splits["x_train"][:4], splits["y_train"][:4]):
    ax.imshow(img)
    ax.set_title("dog" if lbl == 1 else "cat")
    ax.axis("off")
fig.tight_layout()

## 4. Feature extraction: HOG + color histogram
`ImageFeaturePipeline.fit()` is called **only** on the training split; validation/test are
transformed with the already-fitted scaler — preventing train/test leakage.

In [ ]:
from src.features.feature_extractor import ImageFeaturePipeline

feature_pipeline = ImageFeaturePipeline(
    hog_orientations=config.get("features", "hog", "orientations"),
    hog_pixels_per_cell=config.get("features", "hog", "pixels_per_cell"),
    hog_cells_per_block=config.get("features", "hog", "cells_per_block"),
    hog_block_norm=config.get("features", "hog", "block_norm"),
    hist_bins=config.get("features", "color_histogram", "bins_per_channel"),
)

x_train_feat = feature_pipeline.fit_transform(splits["x_train"])
x_val_feat = feature_pipeline.transform(splits["x_val"])
x_test_feat = feature_pipeline.transform(splits["x_test"])
x_train_feat.shape

## 5. Train the SVM

In [ ]:
from src.models.svm_model import SVMClassifier

classifier = SVMClassifier(
    kernel=config.get("model", "kernel"),
    C=config.get("model", "C"),
    gamma=config.get("model", "gamma"),
    probability=config.get("model", "probability"),
    random_seed=config.get("project", "random_seed"),
)
classifier.fit(x_train_feat, splits["y_train"], grid_search=config.get("model", "grid_search"))

## 6. Evaluate on validation and held-out test splits

In [ ]:
val_metrics = classifier.evaluate(x_val_feat, splits["y_val"])
test_metrics = classifier.evaluate(x_test_feat, splits["y_test"])
val_metrics, test_metrics

In [ ]:
import numpy as np

cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["cat", "dog"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["cat", "dog"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im)
fig.tight_layout()

## 7. Persist model + feature pipeline (same artifacts `main.py train` produces)

In [ ]:
from pathlib import Path

model_dir = Path(config.get("paths", "model_dir"))
classifier.save(model_dir / "svm_classifier.joblib")
feature_pipeline.save(model_dir / "feature_pipeline.joblib")
list(model_dir.glob("*.joblib"))

## 8. Reload + predict on new images (inference path)
Equivalent to: `python main.py predict --input data/raw/test1 --output submission.csv`

In [ ]:
from src.pipeline.prediction_pipeline import run_prediction_pipeline

out_path = run_prediction_pipeline(
    input_dir=config.get("paths", "test_dir"),
    config_path="../config/config.yaml",
    output_csv_name="submission.csv",
)
out_path

## Summary

| Stage | Module |
|---|---|
| Synthetic / real data | `src/data/synthetic_generator.py`, `src/data/data_loader.py` |
| Feature extraction | `src/features/feature_extractor.py` (HOG + color histogram, fit/transform separated) |
| Model | `src/models/svm_model.py` (SVC wrapper, optional GridSearchCV) |
| Orchestration | `src/pipeline/training_pipeline.py`, `src/pipeline/prediction_pipeline.py` |
| CLI | `main.py train / predict / generate-data` |

All artifacts (model, scaler, metrics JSON, confusion-matrix figure, logs, submission CSV) are written to
deterministic, project-root-anchored paths under `artifacts/` and `outputs/`.